In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import seaborn as sns

In [68]:
df=pd.read_csv('/content/CTR-prediction.csv')
df.head()

,Daily Time Spent on Site,Age,Area Income,Daily Internet Usage,Ad Topic Line,City,Gender,Country,Timestamp,Clicked on Ad
0,62.26,32.0,69481.85,172.83,Decentralized real-time circuit,Lisafort,Male,Svalbard & Jan Mayen Islands,2016-06-09 21:43:05,0
1,41.73,31.0,61840.26,207.17,Optional full-range projection,West Angelabury,Male,Singapore,2016-01-16 17:56:05,0
2,44.40,30.0,57877.15,172.83,Total 5thgeneration standardization,Reyesfurt,Female,Guadeloupe,2016-06-29 10:50:45,0
3,59.88,28.0,56180.93,207.17,Balanced empowering success,New Michael,Female,Zambia,2016-06-21 14:32:32,0
4,49.21,30.0,54324.73,201.58,Total 5thgeneration standardization,West Richard,Female,Qatar,2016-07-21 10:54:35,1


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Daily Time Spent on Site  10000 non-null  float64
 1   Age                       10000 non-null  float64
 2   Area Income               10000 non-null  float64
 3   Daily Internet Usage      10000 non-null  float64
 4   Ad Topic Line             10000 non-null  object 
 5   City                      10000 non-null  object 
 6   Gender                    10000 non-null  object 
 7   Country                   10000 non-null  object 
 8   Timestamp                 10000 non-null  object 
 9   Clicked on Ad             10000 non-null  int64  
dtypes: float64(4), int64(1), object(5)
memory usage: 781.4+ KB


In [4]:
df['Clicked on Ad'].value_counts()

,count
Clicked on Ad,
0,5083
1,4917


In [5]:
print('Unique values')
print(f'Ad Topic Line: {df['Ad Topic Line'].nunique()}, City {df['City'].nunique()}, Gender: {df['Gender'].nunique()}, Country: {df['Country'].nunique()}')

Unique values
Ad Topic Line: 559, City 521, Gender: 2, Country: 207


In [6]:
X=df.drop('Clicked on Ad',axis=1)
y=df['Clicked on Ad']

In [7]:
df['Timestamp']=pd.to_datetime(df['Timestamp'])
df['hour']=df['Timestamp'].dt.hour
df['day_of_week']=df['Timestamp'].dt.dayofweek
df['month']=df['Timestamp'].dt.month

In [8]:
X=X.drop(['Timestamp','City','Country'],axis=1)

In [9]:
X.head()

,Daily Time Spent on Site,Age,Area Income,Daily Internet Usage,Ad Topic Line,Gender
0,62.26,32.0,69481.85,172.83,Decentralized real-time circuit,Male
1,41.73,31.0,61840.26,207.17,Optional full-range projection,Male
2,44.40,30.0,57877.15,172.83,Total 5thgeneration standardization,Female
3,59.88,28.0,56180.93,207.17,Balanced empowering success,Female
4,49.21,30.0,54324.73,201.58,Total 5thgeneration standardization,Female


In [15]:
num_cols=['Daily Time Spent on Site','Age','Area Income','Daily Internet Usage']

In [60]:
from sklearn.preprocessing import StandardScaler,OneHotEncoder,OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

In [27]:
preprocessor=ColumnTransformer(
    [
        ('num',StandardScaler(),num_cols),
        ('gender',OneHotEncoder(),['Gender']),
        ('text',TfidfVectorizer(max_features=100,stop_words='english'),'Ad Topic Line')
    ]
)

In [28]:
pipeline=Pipeline(
    [
        ('preprocessor',preprocessor),
        ('classifier',LogisticRegression())
    ]
)

In [32]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [33]:
pipeline.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](6,)","['Daily Time Spent on Site','Age','Area Income','Daily Internet Usage', 'Ad Topic Line','Gender']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,6
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('gender', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specify

In [34]:
log_pred=pipeline.predict(X_test)

In [35]:
acc=accuracy_score(y_test,log_pred)
clf=classification_report(y_test,log_pred)
conf=confusion_matrix(y_test,log_pred)
print(f'Accuracy Score: {acc}')
print(f'Classification Report: {clf}')
print(f'Confusion Matrix: {conf}')

Accuracy Score: 0.769
Classification Report:               precision    recall  f1-score   support

           0       0.76      0.80      0.78      1017
           1       0.78      0.74      0.76       983

    accuracy                           0.77      2000
   macro avg       0.77      0.77      0.77      2000
weighted avg       0.77      0.77      0.77      2000

Confusion Matrix: [[813 204]
 [258 725]]


In [63]:
xgb_preprocessor=ColumnTransformer(
    [
        ('num','passthrough',num_cols),
        ('gender',OrdinalEncoder(),['Gender']),
        ('text',TfidfVectorizer(max_features=200,stop_words='english'),'Ad Topic Line')
    ]
)

In [64]:
df.head(1)

,Daily Time Spent on Site,Age,Area Income,Daily Internet Usage,Ad Topic Line,City,Gender,Country,Timestamp,Clicked on Ad,hour,day_of_week,month
0,62.26,32.0,69481.85,172.83,Decentralized real-time circuit,Lisafort,Male,Svalbard & Jan Mayen Islands,2016-06-09 21:43:05,0,21,3,6


In [69]:
X_xgb=df.drop(['Timestamp','City','Country','Clicked on Ad'],axis=1)
y_xgb=df['Clicked on Ad']

In [70]:
pip install Xgboost

In [71]:
from xgboost import XGBClassifier

In [72]:
pipeline=Pipeline([
    ('preprocessor',xgb_preprocessor),
    ('classifier',XGBClassifier(
        n_estimators=250,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='logloss'
    ))
])

In [73]:
X_train,X_test,y_train,y_test=train_test_split(X_xgb,y_xgb,test_size=0.2,random_state=42,stratify=y_xgb)

In [75]:
pipeline.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](6,)","['Daily Time Spent on Site','Age','Area Income','Daily Internet Usage', 'Ad Topic Line','Gender']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,6
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('gender', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specify

In [76]:
xgb_pred=pipeline.predict(X_test)

In [77]:
acc=accuracy_score(y_test,xgb_pred)
clf=classification_report(y_test,xgb_pred)
conf=confusion_matrix(y_test,xgb_pred)
print(f'Accuracy Score: {acc}')
print(f'Classification Report: {clf}')
print(f'Confusion Matrix: {conf}')

Accuracy Score: 0.811
Classification Report:               precision    recall  f1-score   support

           0       0.79      0.85      0.82      1017
           1       0.84      0.77      0.80       983

    accuracy                           0.81      2000
   macro avg       0.81      0.81      0.81      2000
weighted avg       0.81      0.81      0.81      2000

Confusion Matrix: [[869 148]
 [230 753]]


In [78]:
import joblib
joblib.dump(pipeline, 'logistic_regression_pipeline.joblib')
print("Logistic Regression pipeline saved as 'logistic_regression_pipeline.joblib'")

Logistic Regression pipeline saved as 'logistic_regression_pipeline.joblib'


In [79]:
joblib.dump(pipeline, 'xgboost_pipeline.joblib')
print("XGBoost pipeline saved as 'xgboost_pipeline.joblib'")

XGBoost pipeline saved as 'xgboost_pipeline.joblib'


To load these models later, you can use:
```python
loaded_lr_pipeline = joblib.load('logistic_regression_pipeline.joblib')
loaded_xgb_pipeline = joblib.load('xgboost_pipeline.joblib')
```